In [0]:
%pip install faker

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import random
import pandas as pd
from faker import Faker
 
random.seed(42)
Faker.seed(42)
fake = Faker("en_GB")

In [0]:
''' Create schemes, 8 assistance schemes in total'''
schemes = [
    ("SCH01", "ComCare Short-to-Medium Term Assistance", "Financial Assistance"),
    ("SCH02", "ComCare Long-Term Assistance",            "Financial Assistance"),
    ("SCH03", "Childcare Subsidy Top-up",                "Family Support"),
    ("SCH04", "Student Care Fee Assistance",             "Family Support"),
    ("SCH05", "Utilities Support",                       "Living Expenses"),
    ("SCH06", "Transport Support",                       "Living Expenses"),
    ("SCH07", "Home Ownership Support",                  "Housing"),
    ("SCH08", "Employment Assistance",                   "Employment"),
]
df_schemes = pd.DataFrame(schemes, columns=["scheme_id", "scheme_name", "category"])
scheme_ids = df_schemes["scheme_id"].tolist()
display(df_schemes)

scheme_id,scheme_name,category
SCH01,ComCare Short-to-Medium Term Assistance,Financial Assistance
SCH02,ComCare Long-Term Assistance,Financial Assistance
SCH03,Childcare Subsidy Top-up,Family Support
SCH04,Student Care Fee Assistance,Family Support
SCH05,Utilities Support,Living Expenses
SCH06,Transport Support,Living Expenses
SCH07,Home Ownership Support,Housing
SCH08,Employment Assistance,Employment


In [0]:
REGIONS = ["Central", "East", "West", "North", "North-East"]

def make_nric():
    prefix = random.choice(["S", "T"])
    digits = "".join(str(random.randint(0, 9)) for _ in range(7))
    letter = random.choice("ABCDEFGHIZJ")
    return f"{prefix}{digits}{letter}"
 
 
def make_client(client_id):
    return {
        "client_id": client_id,
        "nric": make_nric(),
        "full_name": fake.name(),
        "address": fake.street_address().replace("\n", ", "),
        "region": random.choice(REGIONS),
        "household_income": random.randint(0, 6000),
        "household_size": random.randint(1, 7),
        "updated_date": "2026-10-01",
    }

In [0]:
''' 1,000 clients for Day 1, plus:
    - 30 deliberately problematic rows
      including: 
        - Lower case NRIC
        - Invalid NRIC
        - Missing household income
        - negative household income
        - Extra whitespaces in names
    - 5 exact duplicate rows
'''
day1 = [make_client(i) for i in range(1, 1001)]
df1 = pd.DataFrame(day1)
for n, i in enumerate(random.sample(range(len(df1)), 30)):
    kind = n % 5
    if kind == 0:
        df1.loc[i, "nric"] = df1.loc[i, "nric"].lower()
    elif kind == 1:
        df1.loc[i, "nric"] = df1.loc[i, "nric"][:5]
    elif kind == 2:
        df1.loc[i, "household_income"] = None
    elif kind == 3:
        df1.loc[i, "household_income"] = -500
    else:
        df1.loc[i, "full_name"] = "  " + df1.loc[i, "full_name"] + "  "
 
df1 = pd.concat([df1, df1.sample(5, random_state=1)], ignore_index=True)

display(df1)

client_id,nric,full_name,address,region,household_income,household_size,updated_date
1,S8908386D,William Jennings,"Flat 32, Sian Streets",North,4827.0,3,2026-10-01
2,S2654235B,Samuel Wright,1 Donald Parkways,Central,3112.0,1,2026-10-01
3,T5940781G,James Moore,"Flat 79, Barker Avenue",Central,4522.0,3,2026-10-01
4,T9310341D,Mr Damien Oliver,"Flat 35, Colin Light",Central,3113.0,3,2026-10-01
5,T5255341Z,Eleanor Baker,"Flat 7, Gemma Common",East,4375.0,6,2026-10-01
6,S2764835A,Sean Norton,"Studio 31, Irene Forks",East,262.0,7,2026-10-01
7,T6413953J,Dr Simon Davis,47 Marsden Knoll,North,3241.0,6,2026-10-01
8,T2423884Z,Hollie Phillips,283 Bethany Light,North,4780.0,4,2026-10-01
9,T3287101C,Frances Holmes-Lloyd,"Flat 3, Guy Locks",East,5574.0,4,2026-10-01
10,S6697,Charlotte Kelly-Hewitt,"Flat 76, Wilkins Dam",Central,5584.0,5,2026-10-01


In [0]:
''' Day 2 client snapshot:
    - the original 1,000 clients
    - 50 changed addresses/regions
    - 20 new clients
    - plus 10 deliberately bad NRICs
'''
df2 = pd.DataFrame(day1)
df2["updated_date"] = "2026-10-02"
for i in random.sample(range(len(df2)), 50):
    df2.loc[i, "address"] = fake.street_address().replace("\n", ", ")
    df2.loc[i, "region"] = random.choice(REGIONS)
new_clients = [make_client(i) for i in range(1001, 1021)]
for c in new_clients:
    c["updated_date"] = "2026-10-02"
df2 = pd.concat([df2, pd.DataFrame(new_clients)], ignore_index=True)
for i in random.sample(range(len(df2)), 10):
    df2.loc[i, "nric"] = "BADNRIC"

display(df2)

client_id,nric,full_name,address,region,household_income,household_size,updated_date
1,S8908386D,William Jennings,"Flat 32, Sian Streets",North,4827,3,2026-10-02
2,S2654235B,Samuel Wright,1 Donald Parkways,Central,3112,1,2026-10-02
3,T5940781G,James Moore,"Flat 79, Barker Avenue",Central,4522,3,2026-10-02
4,T9310341D,Mr Damien Oliver,"Flat 35, Colin Light",Central,3113,3,2026-10-02
5,T5255341Z,Eleanor Baker,"Studio 98M, Smith Harbor",West,4375,6,2026-10-02
6,S2764835A,Sean Norton,"Studio 31, Irene Forks",East,262,7,2026-10-02
7,T6413953J,Dr Simon Davis,47 Marsden Knoll,North,3241,6,2026-10-02
8,T2423884Z,Hollie Phillips,283 Bethany Light,North,4780,4,2026-10-02
9,T3287101C,Frances Holmes-Lloyd,"Flat 3, Guy Locks",East,5574,4,2026-10-02
10,S6697848A,Charlotte Kelly-Hewitt,"Flat 76, Wilkins Dam",Central,5584,5,2026-10-02


In [0]:

''' 2,500 assistance cases, plus:
    - 60 deliberately problematic records
        - Including: 
            - negative assistance amount
            - Missing assistance amount
            - orphan client id
            - Unknown scheme
            - Invalid status
            - future application date
    - 5 duplicate case records
'''
STATUSES = ["Approved", "Pending", "Rejected", "Disbursed"]
rows = []
for case_id in range(1, 2501):
    status = random.choices(STATUSES, weights=[30, 15, 15, 40])[0]
    amount = 0.0 if status == "Rejected" else round(random.uniform(50, 1500), 2)
    rows.append({
        "case_id": case_id,
        "client_id": random.randint(1, 1020),
        "scheme_id": random.choice(scheme_ids),
        "application_date": fake.date_between(start_date="-1y", end_date="today").isoformat(),
        "assistance_amount": amount,
        "status": status,
    })
df_assist = pd.DataFrame(rows)

dirty = random.sample(range(len(df_assist)), 60)
for n, i in enumerate(dirty):
    kind = n % 6
    if kind == 0:
        df_assist.loc[i, "assistance_amount"] = -200.0           
    elif kind == 1:
        df_assist.loc[i, "assistance_amount"] = None             
    elif kind == 2:
        df_assist.loc[i, "client_id"] = random.randint(5000, 9999)  
    elif kind == 3:
        df_assist.loc[i, "scheme_id"] = "SCH99"                 
    elif kind == 4:
        df_assist.loc[i, "status"] = random.choice(["approved ", "APPRVD", "unknown"])  
    else:
        df_assist.loc[i, "application_date"] = "2027-12-31"     
 
df_assist = pd.concat([df_assist, df_assist.sample(5, random_state=2)], ignore_index=True)

display(df_assist)

case_id,client_id,scheme_id,application_date,assistance_amount,status
1,982,SCH03,2026-08-27,701.44,Disbursed
2,910,SCH06,2026-05-07,1034.65,Approved
3,1018,SCH05,2025-12-07,655.08,Pending
4,533,SCH06,2026-06-12,109.59,Pending
5,815,SCH08,2026-10-30,228.29,Disbursed
6,812,SCH06,2026-02-22,0.0,Rejected
7,511,SCH03,2025-12-16,1455.73,Approved
8,154,SCH05,2026-11-12,0.0,Rejected
9,311,SCH08,2026-01-18,1013.26,Disbursed
10,16,SCH08,2026-04-02,1209.69,Approved


In [0]:
OUT_DIR = "/Volumes/workspace/bronze/raw_files"
df_schemes.to_csv(f"{OUT_DIR}/schemes.csv", index=False)
df1.to_csv(f"{OUT_DIR}/clients_day1.csv", index=False)
df2.to_csv(f"{OUT_DIR}/clients_day2.csv", index=False)
df_assist.to_csv(f"{OUT_DIR}/assistance.csv", index=False)

In [0]:
for s in ["gold", "silver"]:
    spark.sql(f"DROP SCHEMA IF EXISTS workspace.{s} CASCADE")

for t in spark.sql("SHOW TABLES IN workspace.bronze").collect():
    spark.sql(f"DROP TABLE IF EXISTS workspace.bronze.{t.tableName}")

In [0]:
for s in ["silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{s}")